# langchain_huggingface smoke test

Tests the langchain_huggingface integration package: HuggingFace LLM/chat/embeddings wrappers for LangChain.

In [ ]:
# Example 1: Top-level public API surface.
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
    HuggingFaceEndpointEmbeddings,
    HuggingFacePipeline,
)

exposed = [
    ChatHuggingFace,
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
    HuggingFaceEndpointEmbeddings,
    HuggingFacePipeline,
]
for cls in exposed:
    print(f"{cls.__name__}: {cls.__module__}")

assert all(isinstance(cls, type) for cls in exposed)

In [ ]:
# Example 2: HuggingFaceEndpoint is a LangChain LLM (BaseLanguageModel subclass).
from langchain_core.language_models import BaseLanguageModel
from langchain_huggingface import HuggingFaceEndpoint

# Verify class hierarchy without actually constructing (which would require network).
assert issubclass(HuggingFaceEndpoint, BaseLanguageModel)

# Inspect declared pydantic fields on the wrapper.
fields = HuggingFaceEndpoint.model_fields
expected_fields = {"repo_id", "task", "temperature", "max_new_tokens"}
present = expected_fields & set(fields)
print(f"HuggingFaceEndpoint fields present: {sorted(present)}")
assert "repo_id" in fields

In [ ]:
# Example 3: ChatHuggingFace is a chat-model wrapper over an LLM.
from langchain_core.language_models import BaseChatModel
from langchain_huggingface import ChatHuggingFace

assert issubclass(ChatHuggingFace, BaseChatModel)

# Confirm the wrapper expects an underlying llm field.
fields = ChatHuggingFace.model_fields
print(f"ChatHuggingFace expects fields: {sorted(fields)[:10]}")
assert "llm" in fields

In [ ]:
# Example 4: HuggingFaceEmbeddings is an Embeddings implementation.
from langchain_core.embeddings import Embeddings
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpointEmbeddings

assert issubclass(HuggingFaceEmbeddings, Embeddings)
assert issubclass(HuggingFaceEndpointEmbeddings, Embeddings)

# Embeddings interface requires embed_documents + embed_query methods.
for cls in (HuggingFaceEmbeddings, HuggingFaceEndpointEmbeddings):
    assert callable(getattr(cls, "embed_documents", None))
    assert callable(getattr(cls, "embed_query", None))
    print(f"{cls.__name__}: embed_documents + embed_query present")

In [ ]:
# Example 5: HuggingFacePipeline class + module structure.
from langchain_core.language_models import BaseLanguageModel
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface.llms import HuggingFaceEndpoint as endpoint_via_llms
from langchain_huggingface.chat_models import ChatHuggingFace as chat_via_chat_models
from langchain_huggingface.embeddings import HuggingFaceEmbeddings as emb_via_embeddings

assert issubclass(HuggingFacePipeline, BaseLanguageModel)

# The submodule re-exports should be the same class objects as top-level.
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFaceEmbeddings
assert endpoint_via_llms is HuggingFaceEndpoint
assert chat_via_chat_models is ChatHuggingFace
assert emb_via_embeddings is HuggingFaceEmbeddings

print("langchain_huggingface public API + submodule re-exports verified")